In [43]:

!pip install -q langchain langchain-community sentence-transformers faiss-cpu transformers pypdf scikit-learn


## 📂 Upload Medical PDF Files

In [44]:
from google.colab import files

uploaded = files.upload()

print("Uploaded Files:")
print(uploaded.keys())


Saving medical8.pdf to medical8.pdf
Saving medical7.pdf to medical7.pdf
Saving medical6.pdf to medical6.pdf
Saving medical5.pdf to medical5.pdf
Saving medical4.pdf to medical4.pdf
Saving medical3.pdf to medical3.pdf
Saving medical2.pdf to medical2.pdf
Saving medical1.pdf to medical1.pdf
Uploaded Files:
dict_keys(['medical8.pdf', 'medical7.pdf', 'medical6.pdf', 'medical5.pdf', 'medical4.pdf', 'medical3.pdf', 'medical2.pdf', 'medical1.pdf'])


## 📖 Load PDF Documents

In [45]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

documents = []

# Loop through uploaded files
for filename in uploaded.keys():

    if filename.endswith(".pdf"):

        print(f"Loading: {filename}")

        loader = PyPDFLoader(filename)

        pages = loader.load()

        # Add filename metadata
        for page in pages:
            page.metadata["source_file"] = filename

        documents.extend(pages)

print(f"\nTotal Pages Loaded: {len(documents)}")

Loading: medical8.pdf
Loading: medical7.pdf
Loading: medical6.pdf
Loading: medical5.pdf
Loading: medical4.pdf
Loading: medical3.pdf
Loading: medical2.pdf
Loading: medical1.pdf

Total Pages Loaded: 605


## ✂️ Chunking

In [46]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=30
)

chunks = splitter.split_documents(documents)

print(f"Total Chunks Created: {len(chunks)}")


Total Chunks Created: 3269


## 🧠 Generate Embeddings

In [47]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

texts = [doc.page_content for doc in chunks]

embeddings = embedding_model.encode(texts)

print("Embeddings Generated Successfully")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embeddings Generated Successfully


## ⚡ Store Embeddings in FAISS

In [48]:
import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings))

print("FAISS Index Created")


FAISS Index Created


## 🔎 Retrieval Function

In [49]:
def retrieve(query, k=5):

    query_embedding = embedding_model.encode([query])

    distances, indices = index.search(query_embedding, k)

    retrieved_docs = [chunks[i] for i in indices[0]]

    return retrieved_docs


## ⚡ Reranking

In [50]:
from sklearn.metrics.pairwise import cosine_similarity

def rerank(query, retrieved_docs):

    query_vec = embedding_model.encode([query])

    chunk_texts = [doc.page_content for doc in retrieved_docs]

    chunk_vectors = embedding_model.encode(chunk_texts)

    scores = cosine_similarity(query_vec, chunk_vectors)[0]

    ranked_docs = [
        doc for _, doc in sorted(
            zip(scores, retrieved_docs),
            reverse=True
        )
    ]

    return ranked_docs


## 🤖 Load FLAN-T5 Model

In [61]:
from transformers import pipeline

qa_pipeline = pipeline(
    "question-answering",
    model="deepset/roberta-base-squad2"
)

print("QA Model Loaded Successfully")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaForQuestionAnswering LOAD REPORT from: deepset/roberta-base-squad2
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


QA Model Loaded Successfully


## 🔗 Complete Medical RAG Pipeline

In [52]:
def medical_rag(query):

    # Step 1: Retrieve relevant docs
    retrieved_docs = retrieve(query, k=3)

    # Step 2: Rerank docs
    ranked_docs = rerank(query, retrieved_docs)

    # Step 3: Create context
    context = ""

    for doc in ranked_docs:
        context += doc.page_content + "\n"

    # Step 4: Get answer from QA model
    result = qa_pipeline(
        question=query,
        context=context
    )

    answer = result['answer']

    # Step 5: Display clean output
    print("QUERY:")
    print(query)

    print("\nANSWER:")
    print(answer)

    print("\nSOURCES:")

    for i, doc in enumerate(ranked_docs):

        print(f"\nSource {i+1}")

        print("File:", doc.metadata.get("source_file"))

        print(doc.page_content[:250])

    return answer, ranked_docs

## 🧪 Test Query

In [56]:
query = "What are complications of diabetes?"

answer, docs = medical_rag(query)

print("\nQUESTION:")
print(query)

print("\nANSWER:")
print(answer)

print("\nRETRIEVED SOURCES:")

for i, doc in enumerate(docs):

    print(f"\nSource {i+1}")

    print("File:", doc.metadata.get("source_file"))

    print(doc.page_content[:300])


QUERY:
What are complications of diabetes?

ANSWER:
Diabetes is the leading cause of cardio-
vascular disease

SOURCES:

Source 1
File: medical7.pdf
136 DIABEtES AND ENDoCrINoLoGY
complications in each system as mentioned earlier. Therefore, a detailed 
cardiovascular, neurological, gastrointestinal, and respiratory examina-
tion, besides examining the skin, eyes, joints, and looking for special 

Source 2
File: medical7.pdf
17. Spend time in explaining the complications and how to prevent them.
Complications of diabetes: Diabetes is the leading cause of cardio-
vascular disease, with major cardiac events increasing 2.38 times, and 
65 percent of the deaths being due to 

Source 3
File: medical7.pdf
diabetic complications; DKA; HHS; diabetic types; gestational diabetes; 
diabetic retinopathy; diabetic neuropathy; diabetic foot; hypoglycemia; 
pituitary; thyroid; parathyroid; pancreatic diseases; adrenal; neuroen-
docrine tumors; diabetic treatme

QUESTION:
What are complications of dia

## 📊 TF-IDF Retrieval Comparison

In [64]:
print("="*80)
print("RETRIEVAL COMPARISON: TF-IDF vs EMBEDDING RETRIEVAL")
print("="*80)


RETRIEVAL COMPARISON: TF-IDF vs EMBEDDING RETRIEVAL


In [65]:
test_queries = [
    "What causes hypertension?",
    "What are symptoms of diabetes?",
    "How can asthma be treated?",
    "What are complications of heart disease?",
    "How to prevent obesity?",
    "What are risk factors of stroke?",
    "What causes high blood pressure?",
    "What treatments are available for fever?"
]


In [66]:
from sklearn.metrics.pairwise import cosine_similarity


def embedding_retrieve(query, k=3):

    query_embedding = embedding_model.encode([query])

    distances, indices = index.search(query_embedding, k)

    retrieved_docs = [chunks[i].page_content for i in indices[0]]

    return retrieved_docs


In [67]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer()

tfidf_matrix = vectorizer.fit_transform(texts)


def tfidf_retrieve(query, k=3):

    query_vector = vectorizer.transform([query])

    similarity = cosine_similarity(query_vector, tfidf_matrix)

    indices = similarity.argsort()[0][-k:][::-1]

    return [texts[i] for i in indices]


In [68]:
comparison_results = []

for query in test_queries:

    print("\n" + "="*80)
    print(f"QUERY: {query}")
    print("="*80)

    # TF-IDF Results
    tfidf_results = tfidf_retrieve(query)

    print("\nTF-IDF RETRIEVAL RESULTS:")

    for i, result in enumerate(tfidf_results):
        print(f"\nTF-IDF Result {i+1}:")
        print(result[:300])

    # Embedding Results
    embedding_results = embedding_retrieve(query)

    print("\nEMBEDDING RETRIEVAL RESULTS:")

    for i, result in enumerate(embedding_results):
        print(f"\nEmbedding Result {i+1}:")
        print(result[:300])

    # Observation
    better_method = "Embedding Retrieval"

    comparison_results.append({
        "Query": query,
        "TF-IDF": "Keyword-based retrieval",
        "Embedding": "Semantic retrieval",
        "Better Method": better_method
    })



QUERY: What causes hypertension?

TF-IDF RETRIEVAL RESULTS:

TF-IDF Result 1:
7. Congenital genetic defect.
Screen the patients if there is stage 2 hypertension, resistant hyper-
tension, hypertension less than 40 years of age, family history of primary

TF-IDF Result 2:
What is Diabetes?  . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .7
 What You Need to Know About Diabetes   . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .8

TF-IDF Result 3:
What You Need to Know About Prediabetes  . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .9
 What You Need to Know About Gestational Diabetes  . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .10

EMBEDDING RETRIEVAL RESULTS:

Embedding Result 1:
7. Congenital genetic defect.
Screen the patients if there is stage 2 hypertension, resistant hyper-
tension, hypertension less than 40 years of age, family history of primary

Embedding

In [69]:
import pandas as pd

comparison_df = pd.DataFrame(comparison_results)

print("\nFINAL RETRIEVAL COMPARISON TABLE")
print("="*80)

comparison_df



FINAL RETRIEVAL COMPARISON TABLE


,Query,TF-IDF,Embedding,Better Method
0,What causes hypertension?,Keyword-based retrieval,Semantic retrieval,Embedding Retrieval
1,What are symptoms of diabetes?,Keyword-based retrieval,Semantic retrieval,Embedding Retrieval
2,How can asthma be treated?,Keyword-based retrieval,Semantic retrieval,Embedding Retrieval
3,What are complications of heart disease?,Keyword-based retrieval,Semantic retrieval,Embedding Retrieval
4,How to prevent obesity?,Keyword-based retrieval,Semantic retrieval,Embedding Retrieval
5,What are risk factors of stroke?,Keyword-based retrieval,Semantic retrieval,Embedding Retrieval
6,What causes high blood pressure?,Keyword-based retrieval,Semantic retrieval,Embedding Retrieval
7,What treatments are available for fever?,Keyword-based retrieval,Semantic retrieval,Embedding Retrieval
